# MLflow y selección de modelo

Este notebook crea o reutiliza los tres runs productivos del experimento `invoice-risk` y luego los compara como evidencia para una decisión humana. El entrenamiento se delega al mismo CLI productivo que se puede ejecutar desde terminal; el notebook no copia ni reimplementa su lógica.

## Conexión al Tracking

Antes de ejecutar, levanta el servidor MLflow y configura `MLFLOW_TRACKING_URI` en el kernel. Este notebook usa exactamente esa URI para consultar y registrar los runs.

In [1]:
import os

import mlflow

tracking_uri = os.environ.get("MLFLOW_TRACKING_URI")
if not tracking_uri:
    raise RuntimeError(
        "Preflight MLflow: falta MLFLOW_TRACKING_URI en este kernel. Inicia el servidor compartido, exporta la URI y reinicia el kernel antes de continuar."
    )

mlflow.set_tracking_uri(tracking_uri)
try:
    mlflow.tracking.MlflowClient().search_experiments(max_results=1)
except Exception as error:
    raise RuntimeError(
        f"Preflight MLflow: no se puede conectar a {tracking_uri}. Verifica que el único servidor MLflow compartido esté activo y que la URI sea accesible. Detalle: {error}"
    ) from error
print(f"Backend MLflow compartido accesible: {tracking_uri}")

Backend MLflow compartido accesible: http://127.0.0.1:5000


## Preparar los candidatos productivos

Esta es la ruta preferida para Tracking. Para cada candidato, primero se consulta `invoice-risk` por el mismo `model_type` y `dataset_version`. Si ya existe un run, se reutiliza y se muestra; si no existe, se ejecuta el CLI `python -m invoiceops.ml.train --model ...` con el intérprete del kernel. Cada nueva ejecución registra un run real con métricas, parámetros, tags y artifacts en MLflow.

Por defecto, `Run All` no duplica runs. Cambia la celda marcada solo cuando quieras generar deliberadamente una nueva corrida para los tres candidatos.

In [2]:
# ⚠ MODIFICA ESTADO: cambia a True solo para forzar una nueva corrida de cada candidato.
FORZAR_NUEVAS_CORRIDAS = False

In [3]:
import html
import re
import subprocess
import sys

from IPython.display import HTML, display

CANDIDATOS = ("dummy", "logistic", "random_forest")
DATASET_VERSION = "invoice-risk-v1"


def matching_runs(model_type):
    experiment = mlflow.get_experiment_by_name("invoice-risk")
    if experiment is None:
        return None
    return mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=(
            f"params.model_type = '{model_type}' and params.dataset_version = '{DATASET_VERSION}'"
        ),
    ).sort_values("start_time", ascending=False)


def show_technical_messages(stderr):
    known_patterns = (
        r"^.*Experiment with name 'invoice-risk' does not exist\. Creating a new experiment\.\n?",
        r"^.*Failed to resolve installed pip version\..*\n?",
        r"^.*mlflow/types/utils\.py:\d+: UserWarning: Hint: Inferred schema contains integer column\(s\)\..*\n\s+warnings\.warn\(\n?",
    )
    technical_messages = []
    remaining = stderr
    for pattern in known_patterns:
        matches = re.findall(pattern, remaining, flags=re.MULTILINE)
        technical_messages.extend(matches)
        remaining = re.sub(pattern, "", remaining, flags=re.MULTILINE)
    if technical_messages:
        display(
            HTML(
                "<details><summary>Mensajes técnicos de MLflow ("
                + str(len(technical_messages))
                + ")</summary><pre>"
                + html.escape("".join(technical_messages))
                + "</pre></details>"
            )
        )
    if remaining.strip():
        display(
            HTML(
                "<strong>stderr no reconocido: requiere revisión</strong><pre>"
                + html.escape(remaining)
                + "</pre>"
            )
        )


def show_candidate(model_type, run, estado):
    metricas = ("accuracy", "precision", "recall", "f1", "roc_auc")
    print("=" * 72)
    print(f"Modelo: {model_type}")
    print(f"Run {estado}: {run['run_id']}")
    print(f"Dataset: {run['params.dataset_version']}")
    print(" | ".join(f"{metrica:<10}" for metrica in metricas))
    print("-+-".join("-" * 10 for _ in metricas))
    print(" | ".join(f"{run[f'metrics.{metrica}']:>10.6f}" for metrica in metricas))


for model_type in CANDIDATOS:
    existentes = matching_runs(model_type)
    if not FORZAR_NUEVAS_CORRIDAS and existentes is not None and not existentes.empty:
        show_candidate(model_type, existentes.iloc[0], "reutilizado")
        continue

    resultado = subprocess.run(
        [sys.executable, "-m", "invoiceops.ml.train", "--model", model_type],
        capture_output=True,
        check=False,
        text=True,
    )
    if resultado.returncode:
        raise RuntimeError(
            f"El CLI productivo falló para {model_type} (exit={resultado.returncode}).\n"
            f"stdout:\n{resultado.stdout}\n\nstderr:\n{resultado.stderr}"
        )
    show_technical_messages(resultado.stderr)
    creado = matching_runs(model_type)
    if creado is None or creado.empty:
        raise RuntimeError(f"El CLI terminó sin registrar un run para {model_type}.")
    show_candidate(model_type, creado.iloc[0], "creado")

print("Los candidatos están disponibles como runs reales en invoice-risk.")

Modelo: dummy
Run creado: 88a4f359fd3c4de8987cdf9b569ccc86
Dataset: invoice-risk-v1
accuracy   | precision  | recall     | f1         | roc_auc   
-----------+------------+------------+------------+-----------
  0.790556 |   0.000000 |   0.000000 |   0.000000 |   0.500000


Modelo: logistic
Run creado: d9c39bc230a243bc9095a846d7287e06
Dataset: invoice-risk-v1
accuracy   | precision  | recall     | f1         | roc_auc   
-----------+------------+------------+------------+-----------
  0.800000 |   0.634921 |   0.106101 |   0.181818 |   0.667080


Modelo: random_forest
Run creado: 91214619c9b94267a2308374379b74a6
Dataset: invoice-risk-v1
accuracy   | precision  | recall     | f1         | roc_auc   
-----------+------------+------------+------------+-----------
  0.788333 |   0.486111 |   0.185676 |   0.268714 |   0.643503
Los candidatos están disponibles como runs reales en invoice-risk.


## Comparación inicial

La tabla conserva solo los identificadores, métricas y metadatos necesarios para comparar runs reproducibles. El orden inicial es por `recall` descendente porque dejar pasar una factura que requería revisión es un falso negativo; no obstante, ordenar no equivale a elegir automáticamente un modelo.

In [4]:
experiment = mlflow.get_experiment_by_name("invoice-risk")
if experiment is None:
    raise RuntimeError("No se creó el experimento invoice-risk después de preparar los candidatos.")

runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
if runs.empty:
    raise RuntimeError("El experimento invoice-risk no tiene runs para comparar.")

comparison = (
    runs.loc[
        :,
        [
            "run_id",
            "params.model_type",
            "metrics.accuracy",
            "metrics.precision",
            "metrics.recall",
            "metrics.f1",
            "metrics.roc_auc",
            "params.dataset_version",
            "tags.git_commit",
        ],
    ]
    .rename(
        columns={
            "params.model_type": "model_type",
            "metrics.accuracy": "accuracy",
            "metrics.precision": "precision",
            "metrics.recall": "recall",
            "metrics.f1": "f1",
            "metrics.roc_auc": "roc_auc",
            "params.dataset_version": "dataset_version",
            "tags.git_commit": "git_commit",
        }
    )
    .sort_values("recall", ascending=False, na_position="last")
    .reset_index(drop=True)
)

display(comparison)

,run_id,model_type,accuracy,precision,recall,f1,roc_auc,dataset_version,git_commit
0,91214619c9b94267a2308374379b74a6,random_forest,0.788333,0.486111,0.185676,0.268714,0.643503,invoice-risk-v1,0225019aff6ffd88b67ed340ee2ea00f5acac3a9
1,d9c39bc230a243bc9095a846d7287e06,logistic,0.800000,0.634921,0.106101,0.181818,0.667080,invoice-risk-v1,0225019aff6ffd88b67ed340ee2ea00f5acac3a9
2,88a4f359fd3c4de8987cdf9b569ccc86,dummy,0.790556,0.000000,0.000000,0.000000,0.500000,invoice-risk-v1,0225019aff6ffd88b67ed340ee2ea00f5acac3a9


## La decisión sigue siendo humana

El primer lugar por `recall` es un candidato para discusión, no una selección automática. Antes de elegir, el equipo responsable debe responder:

1. ¿Qué modelo es el mejor candidato para este caso de uso?
2. ¿Qué criterio de selección debe prevalecer: recall, precision, F1, ROC AUC, accuracy o una combinación explícita?
3. ¿Cuál es el coste operativo y de riesgo de los falsos negativos: facturas que requerían revisión y el modelo dejó pasar?
4. ¿Qué trade-offs entre más detecciones, falsos positivos y capacidad de revisión manual acepta el negocio?

Un mayor recall puede reducir falsos negativos, pero normalmente aumenta revisiones manuales y potencialmente falsos positivos. La tabla ordenada muestra evidencia; no define la política ni autoriza una promoción. `Latest` solo describe el orden cronológico de un run; `Best` requiere criterios explícitos y una decisión humana.

In [5]:
candidate = comparison.iloc[0]
print(
    "Candidato para revisión humana: "
    f"{candidate['model_type']} (run_id={candidate['run_id']}, recall={candidate['recall']:.3f})"
)

Candidato para revisión humana: random_forest (run_id=91214619c9b94267a2308374379b74a6, recall=0.186)


## Ver esta ejecución en MLflow

MLflow UI es un **visor del mismo backend** que consultó y actualizó este notebook; no es una copia de los datos. El instructor debe iniciar ese único backend antes de abrir Jupyter. Después abre la URL configurada en este kernel, entra al experimento `invoice-risk` y compara runs, métricas, parámetros, tags y artifacts.

No esperes ver Promotion, aliases o la auditoría de 04/05 aquí: este notebook solo trabaja con Tracking.

In [6]:
print(f"Abre la misma UI configurada en este kernel: {tracking_uri}")
print("Experiments -> invoice-risk -> runs, métricas, parámetros, tags y artifacts.")

Abre la misma UI configurada en este kernel: http://127.0.0.1:5000
Experiments -> invoice-risk -> runs, métricas, parámetros, tags y artifacts.


## Tracking no es Registry

MLflow Tracking registra experimentos, runs, parámetros, métricas y metadatos para compararlos. MLflow Model Registry administra el ciclo de vida de versiones de modelos que ya fueron aprobadas, por ejemplo mediante registro, aliases o etapas según la política del equipo.